In [4]:
import pandas as pd

# Load the dataset (make sure this CSV is in the same directory as your notebook)
df = pd.read_csv("world_health_data.csv")

# Preview the data
display(df.head())
display(df.info())

,country,country_code,year,health_exp,life_expect,maternal_mortality,infant_mortality,neonatal_mortality,under_5_mortality,prev_hiv,inci_tuberc,prev_undernourishment
0,Aruba,ABW,1999,NaN,73.561000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,1999,NaN,51.258874,NaN,88.285642,38.433841,142.506373,6.684793,NaN,NaN
2,Afghanistan,AFG,1999,NaN,54.846000,NaN,94.600000,64.000000,135.800000,0.100000,NaN,NaN
3,Africa Western and Central,AFW,1999,NaN,49.726429,NaN,101.541373,44.733554,173.943151,NaN,NaN,NaN
4,Angola,AGO,1999,NaN,45.386000,NaN,123.500000,51.000000,208.000000,1.300000,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6650 entries, 0 to 6649
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   country                6650 non-null   object 
 1   country_code           6650 non-null   object 
 2   year                   6650 non-null   int64  
 3   health_exp             5167 non-null   float64
 4   life_expect            6190 non-null   float64
 5   maternal_mortality     4893 non-null   float64
 6   infant_mortality       5856 non-null   float64
 7   neonatal_mortality     5856 non-null   float64
 8   under_5_mortality      5856 non-null   float64
 9   prev_hiv               4270 non-null   float64
 10  inci_tuberc            5429 non-null   float64
 11  prev_undernourishment  4805 non-null   float64
dtypes: float64(9), int64(1), object(2)
memory usage: 623.6+ KB


None

In [5]:
# Drop rows with missing critical data
df = df.dropna(subset=["country", "country_code", "year"])

# Convert year to integer
df["year"] = df["year"].astype(int)

# Define sets for classification
region_set = {
    'Arab World', 'Central Europe and the Baltics', 'East Asia & Pacific',
    'East Asia & Pacific (excluding high income)', 'Euro area', 'Europe & Central Asia',
    'Europe & Central Asia (excluding high income)', 'European Union',
    'Latin America & Caribbean', 'Latin America & Caribbean (excluding high income)',
    'Middle East & North Africa', 'Middle East & North Africa (excluding high income)',
    'North America', 'South Asia', 'Sub-Saharan Africa',
    'Sub-Saharan Africa (excluding high income)'
}

income_set = {
    'Low income', 'Lower middle income', 'Low & middle income', 'Middle income',
    'Upper middle income', 'High income'
}

other_set = {
    'Heavily indebted poor countries (HIPC)', 'IBRD only', 'IDA & IBRD total',
    'IDA blend', 'IDA only', 'IDA total', 'Least developed countries: UN classification',
    'OECD members', 'Small states', 'World'
}

# Classify each entry
def classify_entity(name):
    if name in region_set:
        return 'region'
    elif name in income_set:
        return 'income_level'
    elif name in other_set:
        return 'other'
    else:
        return 'country'

# Apply classification
df["entity_type"] = df["country"].apply(classify_entity)

# Sort for time-series operations
df = df.sort_values(by=["country", "year"])

# Forward/backward fill within each entity
df = df.groupby("country").apply(lambda group: group.ffill().bfill()).reset_index(drop=True)

# Save cleaned and classified data
df.to_csv("cleaned_world_health_data.csv", index=False)

# Display classification counts
print("Entity type counts:")
print(df["entity_type"].value_counts())

# Optional preview
print("\nPreview:\n", df.head())

Entity type counts:
entity_type
country         5850
region           400
other            250
income_level     150
Name: count, dtype: int64

Preview:
        country country_code  year  health_exp  life_expect  \
0  Afghanistan          AFG  1999    9.443391       54.846   
1  Afghanistan          AFG  2000    9.443391       55.298   
2  Afghanistan          AFG  2001    9.443391       55.798   
3  Afghanistan          AFG  2002    9.443391       56.454   
4  Afghanistan          AFG  2003    8.941258       57.344   

   maternal_mortality  infant_mortality  neonatal_mortality  \
0              1346.0              94.6                64.0   
1              1346.0              92.0                62.7   
2              1273.0              89.3                61.5   
3              1277.0              86.6                60.2   
4              1196.0              83.7                58.9   

   under_5_mortality  prev_hiv  inci_tuberc  prev_undernourishment entity_type  
0             

/var/folders/cc/nkwtlsz91x7dqc1rw76k6fvw0000gn/T/ipykernel_4013/1757360783.py:47: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("country").apply(lambda group: group.ffill().bfill()).reset_index(drop=True)
